In [1]:
import pandas as pd


json_path = '/home/antares/Tesis_Data/VicomTech/dmd/gE/30/s3/gE_30_s3_2019-03-15T10;56;02+01;00_rgb_ann_distraction.json'

In [2]:
df =  pd.read_json(json_path)
df

,openlabel
metadata,"{'schema_version': '1.0.0', 'name': 'gE_30_s3_..."
objects,"{'0': {'name': '30', 'type': 'driver', 'frame_..."
frames,"{'0': {'objects': {'0': {}}, 'contexts': {'0':..."
frame_intervals,"[{'frame_start': 0, 'frame_end': 1864}]"
ontologies,{'0': 'http://dmd.vicomtech.org/ontology'}
streams,{'face_camera': {'description': 'Frontal face ...
contexts,"{'0': {'name': '', 'type': 'recording_context'..."
actions,"{'0': {'name': '', 'type': 'gaze_on_road/looki..."


In [3]:
import vcd.core as core

# Create empty vcd object
myVCD = core.VCD()

# Load a VCD file
myVCD.load_from_file(json_path)



In [4]:
myVCD.data['openlabel']['metadata']

{'schema_version': '1.0.0',
 'name': 'gE_30_s3_2019-03-15T10;56;02+01;00_distraction',
 'annotator': '0'}

In [6]:
import pandas as pd
from vcd import core

def extraer_actividades_vcd(file_path):
    # 1. Instanciar y cargar el archivo VCD
    vcd_data = core.VCD()
    vcd_data.load_from_file(file_path)

    # 2. Obtener el nombre del video desde los metadatos
    try:
        video_name = vcd_data.data['openlabel']['metadata']['name']
    except KeyError:
        video_name = file_path

    # 3. Extraer las ACCIONES en lugar de los objetos
    acciones = vcd_data.get_actions()

    filas_datos = []

    # 4. Iterar sobre cada acción y sus intervalos de tiempo
    for act_id, act_data in acciones.items():
        # El tipo de acción (ej: 'drinking', 'talking_on_phone', etc.)
        actividad = act_data.get('type', 'Desconocido')

        # Extraemos la lista de intervalos de frames
        intervalos = act_data.get('frame_intervals', [])

        # Por cada segmento temporal en el que ocurre esta acción, creamos una fila
        for intervalo in intervalos:
            frame_inicio = intervalo.get('frame_start')
            frame_fin = intervalo.get('frame_end')

            filas_datos.append({
                'Video_Origen': video_name,
                'ID_Accion': act_id,
                'Actividad': actividad,
                'Frame_Inicio': frame_inicio,
                'Frame_Fin': frame_fin
            })

    # 5. Convertir a DataFrame
    df_actividades = pd.DataFrame(filas_datos)

    return df_actividades

# --- Ejecución ---
# Sustituye con tu archivo real (el que genera la salida que me mostraste)
archivo_json = "tu_archivo.json"

tabla_actividades = extraer_actividades_vcd(json_path)
print(tabla_actividades.to_string())

                                      Video_Origen ID_Accion                             Actividad  Frame_Inicio  Frame_Fin
0   gE_30_s3_2019-03-15T10;56;02+01;00_distraction         0             gaze_on_road/looking_road             0        949
1   gE_30_s3_2019-03-15T10;56;02+01;00_distraction         0             gaze_on_road/looking_road          1002       1354
2   gE_30_s3_2019-03-15T10;56;02+01;00_distraction         0             gaze_on_road/looking_road          1390       1562
3   gE_30_s3_2019-03-15T10;56;02+01;00_distraction         0             gaze_on_road/looking_road          1601       1845
4   gE_30_s3_2019-03-15T10;56;02+01;00_distraction         0             gaze_on_road/looking_road          1848       1864
5   gE_30_s3_2019-03-15T10;56;02+01;00_distraction         1         gaze_on_road/not_looking_road           950       1001
6   gE_30_s3_2019-03-15T10;56;02+01;00_distraction         1         gaze_on_road/not_looking_road          1355       1389
7   gE_3

In [7]:
myVCD.data['openlabel']['actions']

{'0': {'name': '',
  'type': 'gaze_on_road/looking_road',
  'frame_intervals': [{'frame_start': 0, 'frame_end': 949},
   {'frame_start': 1002, 'frame_end': 1354},
   {'frame_start': 1390, 'frame_end': 1562},
   {'frame_start': 1601, 'frame_end': 1845},
   {'frame_start': 1848, 'frame_end': 1864}],
  'ontology_uid': '0',
  'action_data_pointers': {'annotated': {'type': 'text',
    'frame_intervals': [{'frame_start': 0, 'frame_end': 949},
     {'frame_start': 1002, 'frame_end': 1354},
     {'frame_start': 1390, 'frame_end': 1562},
     {'frame_start': 1601, 'frame_end': 1845},
     {'frame_start': 1848, 'frame_end': 1864}]}}},
 '1': {'name': '',
  'type': 'gaze_on_road/not_looking_road',
  'frame_intervals': [{'frame_start': 950, 'frame_end': 1001},
   {'frame_start': 1355, 'frame_end': 1389},
   {'frame_start': 1563, 'frame_end': 1600},
   {'frame_start': 1846, 'frame_end': 1847}],
  'ontology_uid': '0',
  'action_data_pointers': {'annotated': {'type': 'text',
    'frame_intervals': [{'

In [8]:
import pandas as pd
from vcd import core
from pathlib import Path
import os

def extraer_datos_vcd(ruta_archivo):
    """Extrae las anotaciones de un solo archivo VCD."""
    vcd_data = core.VCD()

    try:
        # Cargar el archivo. Convertimos la ruta a string por compatibilidad
        vcd_data.load_from_file(str(ruta_archivo))
    except Exception as e:
        print(f"Error al leer el archivo {ruta_archivo}: {e}")
        return pd.DataFrame() # Devuelve un DataFrame vacío si hay error

    # Intentar obtener el nombre del video
    try:
        video_name = vcd_data.data['openlabel']['metadata']['name']
    except KeyError:
        video_name = os.path.basename(ruta_archivo)

    filas_datos = []

    # 1. Extraer desde la sección "objects"
    objetos = vcd_data.get_objects()
    for obj_id, obj_data in objetos.items():
        actividad = obj_data.get('type', 'Desconocido')
        intervalos = obj_data.get('frame_intervals', [])

        for intervalo in intervalos:
            filas_datos.append({
                'Ruta_Archivo': str(ruta_archivo),
                'Video_Origen': video_name,
                'ID_Anotacion': obj_id,
                'Categoria': 'Objeto',
                'Actividad': actividad,
                'Frame_Inicio': intervalo.get('frame_start'),
                'Frame_Fin': intervalo.get('frame_end')
            })

    # 2. Extraer desde la sección "actions" (por si otros JSON la usan)
    acciones = vcd_data.get_actions()
    if acciones:
        for act_id, act_data in acciones.items():
            actividad = act_data.get('type', 'Desconocido')
            intervalos = act_data.get('frame_intervals', [])

            for intervalo in intervalos:
                filas_datos.append({
                    'Ruta_Archivo': str(ruta_archivo),
                    'Video_Origen': video_name,
                    'ID_Anotacion': act_id,
                    'Categoria': 'Accion',
                    'Actividad': actividad,
                    'Frame_Inicio': intervalo.get('frame_start'),
                    'Frame_Fin': intervalo.get('frame_end')
                })

    return pd.DataFrame(filas_datos)

def procesar_directorio_anidado(directorio_raiz):
    """Busca todos los JSON en el directorio y sus subcarpetas, y los unifica."""
    # rglob('*.json') busca recursivamente todos los archivos terminados en .json
    rutas_json = Path(directorio_raiz).rglob('*.json')

    lista_dataframes = []
    archivos_procesados = 0

    print(f"Iniciando búsqueda en: {directorio_raiz}")

    for ruta in rutas_json:
        # Ejecutamos la función extractora para cada archivo encontrado
        df_temporal = extraer_datos_vcd(ruta)

        # Si el DataFrame no está vacío, lo añadimos a nuestra lista
        if not df_temporal.empty:
            lista_dataframes.append(df_temporal)
            archivos_procesados += 1

    print(f"Se procesaron {archivos_procesados} archivos JSON exitosamente.")

    # Si encontramos y procesamos datos, unimos todos los DataFrames
    if lista_dataframes:
        # concat une todas las tablas una debajo de la otra
        df_final = pd.concat(lista_dataframes, ignore_index=True)
        return df_final
    else:
        print("No se encontraron datos válidos.")
        return pd.DataFrame()

# --- Ejecución del script ---
# Reemplaza esto con la ruta de tu carpeta principal
carpeta_principal = "/home/antares/Tesis_Data/VicomTech/dmd/"

tabla_maestra = procesar_directorio_anidado(carpeta_principal)

if not tabla_maestra.empty:
    # Mostramos las primeras 10 filas para verificar
    print(tabla_maestra.head(10).to_string())

    # Exportar el resultado final a un archivo Excel o CSV
    tabla_maestra.to_csv("anotaciones_completas.csv", index=False)
    print("\n¡Archivo 'anotaciones_completas.csv' guardado con éxito!")

Iniciando búsqueda en: /home/antares/Tesis_Data/VicomTech/dmd/
Se procesaron 1 archivos JSON exitosamente.
                                                                                                  Ruta_Archivo                                    Video_Origen ID_Anotacion Categoria                      Actividad  Frame_Inicio  Frame_Fin
0  /home/antares/Tesis_Data/VicomTech/dmd/gE/30/s3/gE_30_s3_2019-03-15T10;56;02+01;00_rgb_ann_distraction.json  gE_30_s3_2019-03-15T10;56;02+01;00_distraction            0    Objeto                         driver             0       1864
1  /home/antares/Tesis_Data/VicomTech/dmd/gE/30/s3/gE_30_s3_2019-03-15T10;56;02+01;00_rgb_ann_distraction.json  gE_30_s3_2019-03-15T10;56;02+01;00_distraction            0    Accion      gaze_on_road/looking_road             0        949
2  /home/antares/Tesis_Data/VicomTech/dmd/gE/30/s3/gE_30_s3_2019-03-15T10;56;02+01;00_rgb_ann_distraction.json  gE_30_s3_2019-03-15T10;56;02+01;00_distraction            0    Acci